# LoRA results

A rank-8 adapter on qkv+proj -- **196,608 parameters, 0.72%** of the 27M base --
against full SFT on the same base, same data, same budget.

    python basic.py sft --task instruct --lora 8 --lr 3e-3
    python basic.py sft --task joint    --lora 8 --lr 1e-2

## Setup

In [1]:
%load_ext autoreload
%autoreload 2

import json
import math
import sys
from pathlib import Path

root = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)
sys.path.append(str(root / "src"))
sys.path.append(str(root / "src" / "video"))

import torch

from checkpoint import load_checkpoint
from dataset import BinDataset
from evaluate import estimate_loss
from generate import generate
from instruct import Instruct
from lora import load_lora, param_counts
from paths import CKPT_DIR, LOG_DIR
from reverse import Reverse
from tokenizer import ENDOFTEXT

DEV = "cuda" if torch.cuda.is_available() else "cpu"

# a run's jsonl is the archive; a cell re-reads it instead of re-training.
# log stems trail their checkpoint by a second -- the ckpt path is made first.
RUNS = {
    # reverse, 27M TinyStories base
    "rev sft 1e-4":   "sft_reverse_2026-09-22_15-46-43",
    "rev lora 1e-4":  "lora_reverse_2026-09-22_16-33-47",
    "rev lora 1e-3":  "lora_reverse_2026-09-22_16-29-04",
    "rev lora 3e-3":  "lora_reverse_2026-09-22_16-20-30",
    # instruct
    "ins sft 3e-5":   "sft_instruct_2026-09-22_12-57-54",
    "ins sft 1e-4":   "sft_instruct_2026-09-22_17-53-47",
    "ins lora 3e-3":  "lora_instruct_2026-09-22_12-26-52",
    "joint lora 1e-2": "lora_joint_2026-09-22_12-47-06",
    # 600-step lr sweeps, eval_n 8 -- only comp is readable at that n
    "sw ins sft 3e-5":  "sweep_sft_2026-09-22_17-27-02",
    "sw ins sft 1e-4":  "sweep_sft_2026-09-22_17-29-08",
    "sw ins sft 3e-4":  "sweep_sft_2026-09-22_17-31-13",
    "sw ins lora 1e-3": "sweep_lora_2026-09-22_17-33-17",
    "sw ins lora 3e-3": "sweep_lora_2026-09-22_17-35-12",
    "sw ins lora 1e-2": "sweep_lora_2026-09-22_17-37-07",
    "sw joint 3e-4":  "joint_lr_sweep_2026-09-22_12-13-42",
    "sw joint 1e-3":  "joint_lr_sweep_2026-09-22_12-20-41",
    "sw joint 3e-3":  "joint_lr_sweep_2026-09-22_12-27-02",
    "sw joint 1e-2":  "joint_lr_sweep_2026-09-22_12-34-52",
    "sw rev lora 3e-4": "lora_lr_probe_2026-09-22_16-14-25",
    "sw rev lora 1e-3": "lora_lr_probe_2026-09-22_16-15-01",
    "sw rev lora 3e-3": "lora_lr_probe_2026-09-22_16-15-35",
}
BASE = CKPT_DIR / "tinystories_2026-09-22_13-20-13.pt"
CKPTS = {
    "rev sft 1e-4":    CKPT_DIR / "sft_reverse_2026-09-22_15-46-43.pt",
    "rev lora 3e-3":   CKPT_DIR / "lora_reverse_2026-09-22_16-20-30.pt",
    "ins sft 3e-5":    CKPT_DIR / "sft_instruct_2026-09-22_12-57-53.pt",
    "ins sft 1e-4":    CKPT_DIR / "sft_instruct_2026-09-22_17-53-47.pt",
    "ins lora 3e-3":   CKPT_DIR / "lora_instruct_2026-09-22_12-26-51.pt",
    "joint lora 1e-2": CKPT_DIR / "lora_joint_2026-09-22_12-47-02.pt",
}


def load_run(label):
    """(config, eval rows, summary)."""
    text = (LOG_DIR / f"{RUNS[label]}.jsonl").read_text()
    ev = [json.loads(line) for line in text.splitlines()]
    pick = lambda e: [x for x in ev if x["event"] == e]
    return pick("config")[0], pick("log"), pick("summary")[0]


def table(rows, cols, w=12):
    print("".join(f"{c:>{w}}" for c in cols))
    for r in rows:
        print("".join(
            f"{v:>{w}.4f}" if isinstance(v, float) else f"{'-' if v is None else v:>{w}}"
            for v in (r.get(c) for c in cols)))

## 1. the lr does not transfer between tasks

600-step sweeps, eval_n 8 so only `comp` is readable, scored on `comp` **and**
base loss together. **Reads logs, < 1 s.**

Result: each task wants a different lr, and the cost of a hot one differs wildly.

| task     | best lr | note                                        |
| -------- | ------- | ------------------------------------------- |
| reverse  | 3e-3    | at 3e-3 the base loses **+3.80** nats       |
| instruct | 3e-3    | at 3e-3 it loses **+0.115** -- 33x less     |
| joint    | 1e-2    | tolerates the hottest lr of the three       |

Reverse is a mechanism the base has no prior for, so it gets forced into the
attention matmuls -- exactly where the adapter lives. Instruct is a format the
base is already most of the way to. **Sweep per task, never inherit.**

In [2]:
for group, labels in (
    ("instruct sft", ["sw ins sft 3e-5", "sw ins sft 1e-4", "sw ins sft 3e-4"]),
    ("instruct lora", ["sw ins lora 1e-3", "sw ins lora 3e-3", "sw ins lora 1e-2"]),
    ("joint lora", ["sw joint 3e-4", "sw joint 1e-3", "sw joint 3e-3", "sw joint 1e-2"]),
    ("reverse lora", ["sw rev lora 3e-4", "sw rev lora 1e-3", "sw rev lora 3e-3"]),
):
    print(f"--- {group}")
    for label in labels:
        cfg, rows, _ = load_run(label)
        r = rows[-1]
        extra = "  ".join(
            f"{k} {r[k]:.3f}"
            for k in ("exact_match", "reverse_exact_match", "instruct_words")
            if k in r
        )
        print(f"  lr {cfg['lr']:>7.0e}   comp {r['comp']:.4f}   {extra}")
    print()

--- instruct sft
  lr   3e-05   comp 1.1406   
  lr   1e-04   comp 1.1207   
  lr   3e-04   comp 1.1236   

--- instruct lora
  lr   1e-03   comp 1.1664   
  lr   3e-03   comp 1.1547   
  lr   1e-02   comp 1.1527   

--- joint lora
  lr   3e-04   comp 1.0248   reverse_exact_match 0.775  instruct_words 0.558
  lr   1e-03   comp 0.9520   reverse_exact_match 0.975  instruct_words 0.617
  lr   3e-03   comp 0.9266   reverse_exact_match 1.000  instruct_words 0.600
  lr   1e-02   comp 0.9195   reverse_exact_match 1.000  instruct_words 0.708

--- reverse lora
  lr   3e-04   comp 0.4003   exact_match 0.700
  lr   1e-03   comp 0.1620   exact_match 0.965
  lr   3e-03   comp 0.1195   exact_match 1.000



## 2. reverse: LoRA does not win

Full SFT at 1e-4 against the adapter at three lrs, 1000 steps each.
**Reads logs, < 1 s.**

Result: **no lr lets LoRA beat full SFT on both axes.**

| run           | exact_match | comp   | ts val vs base |
| ------------- | ----------- | ------ | -------------- |
| full sft 1e-4 | **1.000**   | 0.0000 | **+0.06**      |
| lora 1e-4     | 0.735       | 0.3194 | +0.12          |
| lora 1e-3     | 1.000       | 0.1179 | +0.48          |
| lora 3e-3     | 1.000       | 0.1175 | +3.80          |

**The comp floor is rank, not lr** -- 0.1179 and 0.1175 across a 3x lr gap is
where rank 8 runs out, and `exact_match` has a ceiling so it cannot show that.
The 1e-4 row sits at 0.3194 and still falling, so that one is undertrained
rather than rank-limited.

In [3]:
for label in ("rev sft 1e-4", "rev lora 1e-4", "rev lora 1e-3", "rev lora 3e-3"):
    cfg, rows, summ = load_run(label)
    solved = next((r["step"] for r in rows if r["exact_match"] == 1.0), None)
    print(f"{label:>14}  lr {cfg['lr']:>7.0e}   1.000 @ {str(solved) if solved else 'never':>5}"
          f"   exact {rows[-1]['exact_match']:.3f}   comp {rows[-1]['comp']:.4f}"
          f"   {summ['total_time_s']:.0f}s")

  rev sft 1e-4  lr   1e-04   1.000 @   300   exact 1.000   comp 0.0000   113s
 rev lora 1e-4  lr   1e-04   1.000 @ never   exact 0.735   comp 0.3194   83s
 rev lora 1e-3  lr   1e-03   1.000 @   700   exact 1.000   comp 0.1179   83s
 rev lora 3e-3  lr   3e-03   1.000 @   400   exact 1.000   comp 0.1175   82s


## 3. instruct and joint: the scorecard

Every model on one scale. Task metrics at **n=100** (the runs logged n=40, which
carries about +/-0.1); base loss on identical val windows.
**Loads six checkpoints, ~5 min.**

Result: **the joint adapter solves reverse for free.**

| model           | trainable | comp   | w_all | rev_em    | vs base   |
| --------------- | --------- | ------ | ----- | --------- | --------- |
| base            | 27.0M     | 1.3986 | 0.010 | 0.000     | -         |
| ins sft 1e-4    | 27.0M     | 1.0945 | 0.420 | 0.000     | +0.2749   |
| ins sft 3e-5    | 27.0M     | 1.1102 | 0.320 | 0.000     | +0.1577   |
| ins lora 3e-3   | 196,608   | 1.1398 | 0.340 | 0.000     | **+0.1334** |
| joint lora 1e-2 | 196,608   | 0.9041 | 0.280 | **1.000** | +0.1574   |

**LoRA's edge is real but small.** Against full SFT tuned for quality (1e-4) it
drifts half as much; against full SFT tuned for gentleness (3e-5) it ties on the
scoreboard and drifts ~15% less. Comparing at one lr overstates it.

**The joint adapter is the result.** `rev_em` 1.000 while its instruct scores sit
within one standard error of the instruct-only adapter (+/-0.05 at n=100) -- one
rank-8 adapter, both skills. The old track got 0.969 with the same adapter size
on a same-size base.

Two cautions: joint `comp` averages both halves and reverse completions are near
zero, so 0.9041 is **not** comparable to the instruct-only rows. And every
`w_all` gap here is under 1 sigma.

In [4]:
ds, gen = BinDataset("val"), torch.Generator()
ins, rev = Instruct(), Reverse()

print(f"{'model':>16} {'trainable':>11} {'step':>5} {'comp':>7} {'words':>6}"
      f" {'w_all':>6} {'stop':>5} {'rev_em':>7} {'ts val':>8} {'vs base':>8}")
base_loss = None
for label, loader, path in (
    ("base", load_checkpoint, BASE),
    ("ins sft 1e-4", load_checkpoint, CKPTS["ins sft 1e-4"]),
    ("ins sft 3e-5", load_checkpoint, CKPTS["ins sft 3e-5"]),
    ("ins lora 3e-3", load_lora, CKPTS["ins lora 3e-3"]),
    ("joint lora 1e-2", load_lora, CKPTS["joint lora 1e-2"]),
):
    m, meta = loader(path, DEV)
    n, total = param_counts(m)
    gen.manual_seed(0)  # identical windows for every row
    loss = estimate_loss(m, ds, 16, 512, iters=50, generator=gen, device=DEV)
    base_loss = loss if base_loss is None else base_loss
    s = ins.evaluate(m, 100)
    print(f"{label:>16} {total if label == 'base' else n:>11,} {meta.get('step', '-'):>5}"
          f" {meta.get('val_loss', float('nan')):>7.4f} {s['words']:>6.3f}"
          f" {s['words_all']:>6.3f} {s['stop_rate']:>5.3f}"
          f" {rev.evaluate(m, 100)['exact_match']:>7.3f}"
          f" {loss:>8.4f} {loss - base_loss:>+8.4f}")
    del m
    torch.cuda.empty_cache()
ds.close()

           model   trainable  step    comp  words  w_all  stop  rev_em   ts val  vs base


            base  27,014,144 33000  1.3986  0.193  0.010 0.610   0.000   1.3563  +0.0000


    ins sft 1e-4  27,014,144  1800  1.0945  0.747  0.420 0.930   0.000   1.6312  +0.2749


    ins sft 3e-5  27,014,144  2000  1.1102  0.720  0.320 0.910   0.000   1.5140  +0.1577


   ins lora 3e-3     196,608  1800  1.1398  0.723  0.340 0.930   0.000   1.4897  +0.1334


 joint lora 1e-2     196,608  2000  0.9041  0.673  0.280 0.910   1.000   1.5137  +0.1574


## 4. what the joint adapter writes

One adapter, both tasks, same weights. **Loads one checkpoint, ~1 min.**

In [5]:
model, meta = load_lora(CKPTS["joint lora 1e-2"], DEV)
model.eval()
print(f"step {meta['step']}   {param_counts(model)[0]:,} trainable\n")

rev = Reverse()
print("--- reverse ---")
for word in ["cat", "zebra", "puzzle"]:
    ids = [int(rev.char_ids[ord(c) - ord("a")]) for c in word] + [rev.sep_id]
    x = torch.tensor([ids], device=DEV)
    out = generate(model, x, len(word) + 1, temperature=0.0)
    got = rev.tok.decode(out[0, x.size(1) :].tolist()).replace(ENDOFTEXT, "<eot>")
    print(f"  {word + '>':<9} -> {got!r}")

task = Instruct()
prompt = task.prompts()[1]
x = torch.tensor([task.tok.encode(prompt)], device=DEV)
print(f"\n--- instruct ---\n{prompt}")
for tag, kw in (("greedy", {"temperature": 0.0}),
                ("t=0.8 p=0.95", {"temperature": 0.8, "top_p": 0.95})):
    kw = dict(kw)
    if kw["temperature"]:
        kw["generator"] = torch.Generator(device=DEV).manual_seed(0)
    out = generate(model, x, 300, use_cache=True, **kw)
    text = task.tok.decode(out[0, x.size(1) :].tolist())
    print(f"\n--- {tag} | stopped {ENDOFTEXT in text}"
          f" | reward {task.reward(prompt, text):.2f} ---")
    print(text.split(ENDOFTEXT)[0])

step 2000   196,608 trainable

--- reverse ---
  cat>      -> 'tac<eot>'
  zebra>    -> 'arbez<eot>'
  puzzle>   -> 'elzzup<eot>'



--- instruct ---
Words: meet, waffle, new
Summary: Lily wants a waffle but gets bitten by a dog while playing with a toy car and has to go to the hospital for stitches.
Story:




--- greedy | stopped True | reward 0.67 ---
Once upon a time, there was a little girl named Lily. She loved waffles and ate them every day for breakfast. One day, Lily's mom made her a new waffle for breakfast. It was so yummy and yummy!
Lily wanted to eat it all by herself, but her mom said she had to wait until after breakfast. Lily was sad, but she knew she had to wait. She went to the kitchen and saw a big dog playing with a toy car. She wanted to play with it too, but her mom said she had to wait until after breakfast.
Lily was still sad, but she knew she had to wait. She went back to the kitchen and saw a little boy playing with a toy car. She asked him if she could play with it too. The little boy said yes and they played together. Lily was happy again and forgot about the waffle. She was glad she waited and got to play with the toy car.



--- t=0.8 p=0.95 | stopped True | reward 0.67 ---
Once upon a time, there was a little girl named Lily. She was very excited because her mommy was going to introduce her to a new kind of waffle.
Lily was very excited to try a waffle. She took a big bite and it tasted even better than it looked! She was very happy and ate the whole waffle in one bite.
After she finished her waffle, Lily went outside to play with her toy car. She had so much fun playing with her car that she didn't even notice the time passing by. When it was time to go home, she said goodbye to her new friend and promised to come back soon.
